# 00 · ODD 与 L4 系统契约

L4 不是单个神经网络的准确率等级，而是一个自动驾驶系统在明确的 **Operational Design Domain（ODD）** 内完成任务、检测退化并安全退出的系统属性。本 notebook 先把“模型输入输出”放进可验证的系统契约中。

学习目标：

- 用结构化字段定义 ODD，而不是用“城市道路”这种模糊描述；
- 为 sensor bundle、localization、planner 和 safety monitor 写输入/输出/时间约束；
- 区分 `NOMINAL`、`DEGRADED`、`MINIMAL_RISK` 与 `ODD_EXIT`；
- 计算 ODD coverage、契约违规率和延迟预算余量。

下面的实现是可运行的教学版。它不构成真实车辆的 safety case，也不能替代 ISO 26262、SOTIF 或公司的系统安全流程。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass
from ipywidgets import interact, FloatSlider

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True


## Part A — 把 ODD 写成可计算的约束

这里定义一个简化的 robotaxi ODD：限定地理区域、最高速度、允许的雨量、是否允许夜间运行，以及定位和传感器的最低健康度。真实系统还会加入道路类型、施工、地图版本、交通规则、通信和车辆平台等约束。


In [ ]:
@dataclass(frozen=True)
class ODD:
    geography: str = "urban_mapped"
    max_speed_mps: float = 13.9
    max_rain_mm_h: float = 8.0
    night_allowed: bool = False
    min_sensor_health: float = 0.70
    max_localization_sigma_m: float = 0.80


odd = ODD()


def make_scenarios(n=320, seed=7):
    rng = np.random.default_rng(seed)
    return pd.DataFrame(
        {
            "geography": rng.choice(["urban_mapped", "rural_unmapped", "urban_mapped"], n),
            "speed_mps": rng.uniform(4.0, 18.0, n),
            "rain_mm_h": rng.uniform(0.0, 14.0, n),
            "night": rng.random(n) < 0.25,
            "sensor_health": rng.uniform(0.45, 1.0, n),
            "localization_sigma_m": rng.uniform(0.15, 1.35, n),
        }
    )


def classify_odd(frame, definition=odd):
    checks = pd.DataFrame(
        {
            "geography_ok": frame["geography"].eq(definition.geography),
            "speed_ok": frame["speed_mps"].le(definition.max_speed_mps),
            "rain_ok": frame["rain_mm_h"].le(definition.max_rain_mm_h),
            "light_ok": frame["night"].le(definition.night_allowed),
            "sensor_ok": frame["sensor_health"].ge(definition.min_sensor_health),
            "localization_ok": frame["localization_sigma_m"].le(definition.max_localization_sigma_m),
        }
    )
    return checks, checks.all(axis=1)


scenarios = make_scenarios()
checks, scenarios["in_odd"] = classify_odd(scenarios)
coverage = scenarios["in_odd"].mean()
print(f"synthetic ODD coverage: {coverage:.1%}")
display(scenarios.head())


In [ ]:
check_rate = checks.mean().sort_values()
ax = check_rate.plot(kind="barh", color="#4f8cc9")
ax.set_xlim(0, 1)
ax.set_xlabel("fraction satisfying the constraint")
ax.set_title("Which ODD constraint removes the most scenarios?")
plt.show()


## Part B — 为模块定义输入、输出和时间契约

一个模型接口至少应说明：

- 输入字段和单位；
- 时间戳、最大允许 age 和坐标系；
- 输出的语义及其置信度；
- 最大 latency 和缺失数据处理方式。

`validate_sensor_bundle` 只做静态检查；它不会证明传感器数据正确，也不会证明模型在 OOD 场景中安全。


In [ ]:
SENSOR_CONTRACT = {
    "required_keys": {"camera", "lidar", "imu", "timestamp_s", "frame_id"},
    "frame_id": "base_link",
    "max_age_s": {"camera": 0.15, "lidar": 0.10, "imu": 0.05},
    "max_latency_ms": 100.0,
}


def validate_sensor_bundle(bundle, contract=SENSOR_CONTRACT):
    issues = []
    missing = contract["required_keys"] - set(bundle)
    if missing:
        issues.append(f"missing keys: {sorted(missing)}")
    if bundle.get("frame_id") != contract["frame_id"]:
        issues.append("frame_id does not match the module contract")
    for sensor, max_age in contract["max_age_s"].items():
        age = bundle.get(f"{sensor}_age_s")
        if age is None:
            issues.append(f"missing age for {sensor}")
        elif age > max_age:
            issues.append(f"{sensor} age {age:.3f}s exceeds {max_age:.3f}s")
    latency = bundle.get("latency_ms")
    if latency is not None and latency > contract["max_latency_ms"]:
        issues.append(f"latency {latency:.1f}ms exceeds budget")
    return {"valid": not issues, "issues": issues}


valid_bundle = {
    "camera": np.zeros((8, 8, 3)), "lidar": np.zeros((20, 4)), "imu": np.zeros(6),
    "timestamp_s": 12.0, "frame_id": "base_link", "camera_age_s": 0.04,
    "lidar_age_s": 0.03, "imu_age_s": 0.01, "latency_ms": 61.0,
}
broken_bundle = {**valid_bundle, "frame_id": "camera_front", "lidar_age_s": 0.21, "latency_ms": 124.0}
print("valid:", validate_sensor_bundle(valid_bundle))
print("broken:", validate_sensor_bundle(broken_bundle))


## Part C — 从健康度到系统状态

L4 系统需要把模型输出和运行时健康度连接起来。下面的状态机是最小示例：

- `NOMINAL`：正常运行；
- `DEGRADED`：降低速度或切换到保守策略；
- `MINIMAL_RISK`：执行最小风险动作；
- `ODD_EXIT`：当前条件超出 ODD，停止接受新的自动驾驶任务。


In [ ]:
def system_state(sensor_health, localization_sigma, latency_ms, odd_ok,
                 min_health=0.70, max_sigma=0.80, max_latency=100.0):
    if not odd_ok:
        return "ODD_EXIT"
    if sensor_health < 0.45 or localization_sigma > 1.50 or latency_ms > 1.5 * max_latency:
        return "MINIMAL_RISK"
    if sensor_health < min_health or localization_sigma > max_sigma or latency_ms > max_latency:
        return "DEGRADED"
    return "NOMINAL"


health = np.linspace(0.95, 0.35, 80)
sigma = np.linspace(0.20, 1.70, 80)
latency = np.linspace(55.0, 140.0, 80)
states = [
    system_state(h, s, l, odd_ok=(i < 68))
    for i, (h, s, l) in enumerate(zip(health, sigma, latency))
]
print(pd.Series(states).value_counts().to_dict())

state_code = {"NOMINAL": 0, "DEGRADED": 1, "MINIMAL_RISK": 2, "ODD_EXIT": 3}
plt.step(np.arange(len(states)), [state_code[s] for s in states], where="post")
plt.yticks(list(state_code.values()), list(state_code))
plt.xlabel("cycle")
plt.ylabel("system state")
plt.title("A system state is not the same thing as a model confidence score")
plt.show()


### 交互练习

调整最低传感器健康度，观察 ODD coverage 和状态分布如何变化。然后完成以下 TODO：

1. 增加 `map_version` 和 `route_available` 两个契约字段；
2. 给 `DEGRADED` 增加恢复滞回（连续若干个健康周期后才回到 `NOMINAL`）；
3. 计算每条约束单独造成的 coverage loss；
4. 写出一个“模型置信度很高，但输入已经 stale”的反例。


In [ ]:
def inspect_odd(min_sensor_health=0.70, max_localization_sigma=0.80):
    definition = ODD(
        min_sensor_health=min_sensor_health,
        max_localization_sigma_m=max_localization_sigma,
    )
    _, in_odd = classify_odd(scenarios, definition)
    print(f"coverage = {in_odd.mean():.1%}")
    print(pd.Series([
        system_state(row.sensor_health, row.localization_sigma_m, 70.0, bool(ok),
                     min_health=min_sensor_health,
                     max_sigma=max_localization_sigma)
        for row, ok in zip(scenarios.itertuples(), in_odd)
    ]).value_counts().to_dict())


interact(
    inspect_odd,
    min_sensor_health=FloatSlider(value=0.70, min=0.45, max=0.95, step=0.05),
    max_localization_sigma=FloatSlider(value=0.80, min=0.30, max=1.50, step=0.05),
)


## 完成标准

你应能提交一页 system contract：列出 ODD、每个模块的输入输出、时间预算、坐标系、降级动作和验证指标。只写“模型准确率 95%”不能替代这份契约。
